In [ ]:
# === NORTHSTAR -- Step [0] of Pipeline ===
# Tower 7: Tower of Kids -- Kids Safety QA for solace-browser
# Every probe checks REAL code. No mocks, no fakes.

NORTHSTAR = "kids-safety-qa"
NOTEBOOK_PATH = "notebooks/qa/02-kids-safety-qa.ipynb"
TOWER = 7
TOWER_NAME = "Tower of Kids"
TOTAL_PROBES = 16
MIN_SCORE = 70

# Tower DNA: nextgen(product) = intuition(no_manual) x safety(absolute) x fun(genuine) x growth(learning)
# 49 floors, 343 questions, 49 kid personas ages 10-17, target: solace-browser

import os
import sys
import json
import re
import ast
import importlib
from pathlib import Path
from collections import Counter

PROJECT_ROOT = "/home/phuc/projects/solace-browser"
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
WEB_DIR = os.path.join(PROJECT_ROOT, "web")
CSS_FILE = os.path.join(WEB_DIR, "css", "site.css")
LOCALES_DIR = os.path.join(PROJECT_ROOT, "app", "locales", "yinyang")
RECIPES_DIR = os.path.join(PROJECT_ROOT, "data", "default", "recipes")
OAUTH3_DIR = os.path.join(SRC_DIR, "oauth3")

HTML_FILES = [
    "home.html", "settings.html", "app-store.html", "download.html",
    "docs.html", "demo.html", "start.html", "schedule.html",
    "tunnel-connect.html", "machine-dashboard.html", "style-guide.html",
    "create-app.html", "app-detail.html",
    "partials-header.html", "partials-footer.html",
    "docs/quick-start.html", "docs/oauth3.html", "docs/mcp.html",
]

# Add src to path for imports
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

results = {}  # probe_name -> {"pass": bool, "detail": str}

print(f"NORTHSTAR: {NORTHSTAR}")
print(f"TOWER: {TOWER} -- {TOWER_NAME}")
print(f"PROBES: {TOTAL_PROBES}")
print(f"MIN_SCORE: {MIN_SCORE}%")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"HTML surfaces: {len(HTML_FILES)} files")

In [ ]:
# === PROBE 1: COPPA Age Gate (Floor 4 Jayden, Floor 48 Dragon Rider) ===
# Q: Does an age gate prevent a 10-year-old from signing up without parental consent?
# Q: Is the product COPPA compliant -- could a 10-year-old sign up right now?
# Check: ws_bridge.py must have content filter that detects minors under 13

probe_name = "coppa_age_gate"
passed = True
details = []

ws_bridge_path = os.path.join(SRC_DIR, "yinyang", "ws_bridge.py")
assert os.path.exists(ws_bridge_path), f"ws_bridge.py not found at {ws_bridge_path}"

ws_source = Path(ws_bridge_path).read_text(encoding="utf-8")

# Check 1: COPPA comment/reference exists
has_coppa_ref = "COPPA" in ws_source or "coppa" in ws_source
if not has_coppa_ref:
    passed = False
    details.append("FAIL: No COPPA reference in ws_bridge.py")
else:
    details.append("PASS: COPPA reference found in ws_bridge.py")

# Check 2: Age detection pattern exists (regex for "I am X years old")
has_age_pattern = "_MINOR_PATTERN" in ws_source or re.search(r"age.*<\s*13", ws_source)
if not has_age_pattern:
    passed = False
    details.append("FAIL: No minor age detection pattern")
else:
    details.append("PASS: Minor age detection pattern found")

# Check 3: Under-13 blocking logic
has_under_13_block = "age < 13" in ws_source or "under 13" in ws_source.lower()
if not has_under_13_block:
    passed = False
    details.append("FAIL: No under-13 blocking logic")
else:
    details.append("PASS: Under-13 blocking logic present")

# Check 4: The blocked patterns list exists (content filter)
has_blocked_patterns = "_BLOCKED_PATTERNS" in ws_source
if not has_blocked_patterns:
    passed = False
    details.append("FAIL: No _BLOCKED_PATTERNS content filter")
else:
    details.append("PASS: Content filter patterns present")

# Check 5: PII solicitation protection for minors
has_pii_protection = "address" in ws_source and "phone" in ws_source and "school" in ws_source
if not has_pii_protection:
    passed = False
    details.append("FAIL: No PII solicitation protection for minors")
else:
    details.append("PASS: PII solicitation protection for minors exists")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 2: Content Safety Filters (Floor 27 Zuri, Floor 6 Mia) ===
# Q: Are there content warnings before potentially triggering content?
# Q: Does the product have a safe browsing mode that filters distressing content?
# Check: ws_bridge.py blocked patterns cover threats, self-harm, profanity

probe_name = "content_safety_filters"
passed = True
details = []

ws_source = Path(ws_bridge_path).read_text(encoding="utf-8")

# Required safety categories in the content filter
safety_categories = {
    "profanity": ["fuck", "shit", "bitch"],
    "threats": ["kill", "bomb", "shoot"],
    "self_harm": ["harm", "kill.*myself", "yourself"],
    "pii_solicitation": ["address", "phone", "school"],
}

for category, keywords in safety_categories.items():
    found = any(kw in ws_source.lower() for kw in keywords)
    if not found:
        passed = False
        details.append(f"FAIL: Missing content filter category: {category}")
    else:
        details.append(f"PASS: Content filter covers {category}")

# Check: filter function returns rejection message (not silent swallow)
has_rejection_msg = "filtered" in ws_source.lower() or "blocked" in ws_source.lower()
if not has_rejection_msg:
    passed = False
    details.append("FAIL: No user-facing rejection message for filtered content")
else:
    details.append("PASS: User-facing rejection message present")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 3: Budget Protection for Kids (Floor 48 Dragon Rider) ===
# Q: Can parents see what their teen is doing with this product?
# Q: Does budget_gates.py enforce spending limits fail-closed?
# Check: budget_gates.py must be fail-closed with no silent fallbacks

probe_name = "budget_protection"
passed = True
details = []

budget_path = os.path.join(SRC_DIR, "budget_gates.py")
assert os.path.exists(budget_path), f"budget_gates.py not found"

budget_source = Path(budget_path).read_text(encoding="utf-8")
tree = ast.parse(budget_source)

# Check 1: Fail-closed design (must have "FAIL-CLOSED" or "fail-closed" in docstring)
has_fail_closed = "fail-closed" in budget_source.lower() or "fail_closed" in budget_source.lower()
if not has_fail_closed:
    passed = False
    details.append("FAIL: budget_gates.py missing fail-closed design")
else:
    details.append("PASS: Fail-closed design documented")

# Check 2: No broad except Exception patterns (Fallback Ban)
broad_except_count = 0
for node in ast.walk(tree):
    if isinstance(node, ast.ExceptHandler):
        if node.type is None:  # bare except:
            broad_except_count += 1
        elif isinstance(node.type, ast.Name) and node.type.id == "Exception":
            broad_except_count += 1

if broad_except_count > 0:
    passed = False
    details.append(f"FAIL: {broad_except_count} broad except handler(s) found (Fallback Ban violation)")
else:
    details.append("PASS: No broad except handlers (Fallback Ban compliant)")

# Check 3: Budget exhaustion raises an error (not returns None)
has_exhaustion_error = "BudgetExhaustedError" in budget_source
if not has_exhaustion_error:
    passed = False
    details.append("FAIL: No BudgetExhaustedError class")
else:
    details.append("PASS: BudgetExhaustedError defined")

# Check 4: Remaining runs check exists
has_remaining_check = "remaining_runs" in budget_source
if not has_remaining_check:
    passed = False
    details.append("FAIL: No remaining_runs check in budget gates")
else:
    details.append("PASS: remaining_runs budget check present")

# Check 5: Step-up required for high-risk actions
has_step_up = "StepUpRequiredError" in budget_source or "step_up" in budget_source.lower()
if not has_step_up:
    passed = False
    details.append("FAIL: No step-up gate for high-risk actions")
else:
    details.append("PASS: Step-up gate for high-risk actions present")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 4: i18n Locale Coverage (Floor 10 Jin, Floor 17 Diego, Floor 25 Fatima) ===
# Q: Is there a Chinese-language interface? Spanish? Arabic?
# Q: Does the product serve 1.4B Chinese speakers or only English speakers?
# Check: locale files exist for zh, es, ar + i18n.py _SUPPORTED set

probe_name = "i18n_locale_coverage"
passed = True
details = []

# Required locales for kids tower personas
required_locales = {
    "zh": "Chinese (Jin, Floor 10)",
    "es": "Spanish (Diego, Floor 17)",
    "ar": "Arabic (Fatima, Floor 25; Aisha, Floor 20)",
    "ko": "Korean",
    "hi": "Hindi (Sana, Floor 14)",
    "ja": "Japanese",
    "vi": "Vietnamese",
    "fr": "French",
    "de": "German",
}

# Check locale JSON files exist
for locale_code, persona_desc in required_locales.items():
    locale_file = os.path.join(LOCALES_DIR, f"{locale_code}.json")
    if os.path.exists(locale_file):
        # Check file is non-empty valid JSON
        try:
            data = json.loads(Path(locale_file).read_text(encoding="utf-8"))
            # Count leaf keys recursively (locale files use nested structure)
            def _count_leaves(obj):
                if isinstance(obj, dict):
                    return sum(_count_leaves(v) for v in obj.values())
                return 1
            key_count = _count_leaves(data)
            if key_count < 5:
                passed = False
                details.append(f"FAIL: {locale_code}.json has only {key_count} keys ({persona_desc})")
            else:
                details.append(f"PASS: {locale_code}.json has {key_count} keys ({persona_desc})")
        except json.JSONDecodeError:
            passed = False
            details.append(f"FAIL: {locale_code}.json is invalid JSON")
    else:
        passed = False
        details.append(f"FAIL: {locale_code}.json missing ({persona_desc})")

# Check i18n.py _SUPPORTED set includes these locales
i18n_path = os.path.join(SRC_DIR, "i18n.py")
i18n_source = Path(i18n_path).read_text(encoding="utf-8")
for locale_code in required_locales:
    if f'"{locale_code}"' in i18n_source:
        details.append(f"PASS: {locale_code} in i18n._SUPPORTED")
    else:
        passed = False
        details.append(f"FAIL: {locale_code} not in i18n._SUPPORTED")

# Check RTL locales are declared
if '"ar"' in i18n_source and "_RTL_LOCALES" in i18n_source:
    details.append("PASS: Arabic RTL locale support declared")
else:
    passed = False
    details.append("FAIL: Arabic RTL locale support missing")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 5: Sensory Safety -- Reduced Motion + Dark Mode (Floor 26 Isla, Floor 5 Kai) ===
# Q: Is there a quiet mode that reduces animations, notifications, and sensory input?
# Q: Does the product have a dark mode toggle and reduced-motion mode?
# Check: CSS has prefers-reduced-motion and prefers-color-scheme media queries

probe_name = "sensory_safety"
passed = True
details = []

assert os.path.exists(CSS_FILE), f"site.css not found at {CSS_FILE}"
css_source = Path(CSS_FILE).read_text(encoding="utf-8")

# Check 1: prefers-reduced-motion media query
reduced_motion_count = len(re.findall(r"prefers-reduced-motion", css_source))
if reduced_motion_count == 0:
    passed = False
    details.append("FAIL: No prefers-reduced-motion media query in site.css")
else:
    details.append(f"PASS: prefers-reduced-motion found {reduced_motion_count} time(s)")

# Check 2: animation: none or transition: none inside reduced-motion block
reduced_blocks = re.findall(
    r"@media\s*\(prefers-reduced-motion:\s*reduce\)\s*\{([^}]+(?:\{[^}]*\}[^}]*)*)\}",
    css_source, re.DOTALL
)
if reduced_blocks:
    block_text = " ".join(reduced_blocks)
    disables_animation = "animation" in block_text.lower() or "transition" in block_text.lower()
    if disables_animation:
        details.append("PASS: Reduced motion block disables animation/transition")
    else:
        passed = False
        details.append("FAIL: Reduced motion block does not disable animation/transition")
else:
    passed = False
    details.append("FAIL: No reduced-motion block content found")

# Check 3: Light mode / color scheme support
has_light_mode = "prefers-color-scheme" in css_source
if has_light_mode:
    details.append("PASS: prefers-color-scheme media query found")
else:
    passed = False
    details.append("FAIL: No prefers-color-scheme support (no light mode toggle)")

# Check schedule.css too
schedule_css = os.path.join(WEB_DIR, "css", "schedule.css")
if os.path.exists(schedule_css):
    sched_source = Path(schedule_css).read_text(encoding="utf-8")
    if "prefers-reduced-motion" in sched_source:
        details.append("PASS: schedule.css also has reduced-motion support")
    else:
        details.append("INFO: schedule.css lacks reduced-motion (minor gap)")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 6: Keyboard Navigation + Skip Nav (Floor 30 Dmitri, Floor 31 Rosa) ===
# Q: Is screen reader compatibility complete in the browser interface?
# Q: Are all UI elements keyboard-navigable?
# Check: skip-nav link injected, ARIA labels on interactive elements in HTML

probe_name = "keyboard_a11y_skip_nav"
passed = True
details = []

# Check 1: skip-nav in CSS
if "skip-nav" in css_source:
    details.append("PASS: .skip-nav CSS class found")
else:
    passed = False
    details.append("FAIL: .skip-nav CSS class missing")

# Check 2: skip-nav injection in layout.js
layout_js_path = os.path.join(WEB_DIR, "js", "layout.js")
if os.path.exists(layout_js_path):
    layout_js = Path(layout_js_path).read_text(encoding="utf-8")
    if "skip-nav" in layout_js:
        details.append("PASS: skip-nav injected via layout.js")
    else:
        passed = False
        details.append("FAIL: layout.js missing skip-nav injection")
else:
    passed = False
    details.append("FAIL: layout.js not found")

# Check 3: ARIA labels in HTML files
aria_count = 0
role_count = 0
alt_count = 0
files_checked = 0

for html_file in HTML_FILES:
    html_path = os.path.join(WEB_DIR, html_file)
    if os.path.exists(html_path):
        files_checked += 1
        content = Path(html_path).read_text(encoding="utf-8")
        aria_count += len(re.findall(r'aria-', content))
        role_count += len(re.findall(r'role=', content))
        alt_count += len(re.findall(r'alt=', content))

details.append(f"INFO: Checked {files_checked} HTML files")
details.append(f"INFO: ARIA attributes: {aria_count}, role attributes: {role_count}, alt attributes: {alt_count}")

if aria_count < 10:
    passed = False
    details.append(f"FAIL: Only {aria_count} ARIA attributes across all HTML -- insufficient for screen readers")
else:
    details.append(f"PASS: {aria_count} ARIA attributes found across HTML surfaces")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 7: Identity + Representation (Floor 12 Ava, Floor 18 Priya, Floor 21 River) ===
# Q: Does the profile system use binary gender fields or support trans identity?
# Q: Are there pronoun settings (he/him, she/her, they/them, custom)?
# Q: Is there ANY representation of non-binary identity anywhere in the product?
# Check: profiles/manager.py for gender/pronoun fields, codebase-wide search

probe_name = "identity_representation"
passed = True
details = []

profile_path = os.path.join(SRC_DIR, "profiles", "manager.py")
assert os.path.exists(profile_path), "profiles/manager.py not found"

profile_source = Path(profile_path).read_text(encoding="utf-8")

# Check 1: Gender field in profile dataclass
has_gender_field = "gender" in profile_source.lower()
if has_gender_field:
    details.append("PASS: Gender-related field found in profiles")
else:
    passed = False
    details.append("FAIL: No gender field in profile dataclass -- identity invisible")

# Check 2: Pronoun support anywhere in codebase
pronoun_found = False
for root, dirs, files in os.walk(SRC_DIR):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    for f in files:
        if f.endswith(".py"):
            content = Path(os.path.join(root, f)).read_text(encoding="utf-8", errors="ignore")
            if re.search(r"pronoun|they/them|non.?binary|gender.?neutral", content, re.I):
                pronoun_found = True
                details.append(f"PASS: Pronoun/non-binary reference in {f}")
                break
    if pronoun_found:
        break

if not pronoun_found:
    passed = False
    details.append("FAIL: No pronoun or non-binary reference in any Python source")

# Check 3: Chosen name vs legal name support
has_chosen_name = "chosen_name" in profile_source or "display_name" in profile_source
has_name_field = "name" in profile_source
if has_chosen_name:
    details.append("PASS: Chosen/display name field in profile")
else:
    details.append(f"INFO: Profile has 'name' but no explicit chosen_name/display_name distinction")

# Check 4: Scan HTML for inclusive language
inclusive_terms_found = 0
for html_file in HTML_FILES:
    html_path = os.path.join(WEB_DIR, html_file)
    if os.path.exists(html_path):
        content = Path(html_path).read_text(encoding="utf-8", errors="ignore")
        if re.search(r"inclusive|diversity|they|identity|pronoun", content, re.I):
            inclusive_terms_found += 1

if inclusive_terms_found > 0:
    details.append(f"PASS: Inclusive language found in {inclusive_terms_found} HTML file(s)")
else:
    details.append("INFO: No explicit inclusive language in HTML surfaces")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 8: Approval Gate Safety (Floor 6 Mia, Floor 48 Dragon Rider) ===
# Q: Is the approval dialog anxiety-inducing or reassuring?
# Q: Does the language provide reassurance that mistakes are reversible?
# Q: Does the auto-deny timer discriminate against users with motor impairments?
# Check: OAuth3 consent_ui.py and approval flow for teen-safe language

probe_name = "approval_gate_safety"
passed = True
details = []

consent_path = os.path.join(OAUTH3_DIR, "consent_ui.py")
assert os.path.exists(consent_path), "consent_ui.py not found"

consent_source = Path(consent_path).read_text(encoding="utf-8")

# Check 1: Consent UI exists and has routes
has_consent_route = "/consent" in consent_source
if has_consent_route:
    details.append("PASS: /consent route defined")
else:
    passed = False
    details.append("FAIL: No /consent route")

# Check 2: Risk level descriptions exist (so teens understand what they're approving)
has_risk_levels = "risk_level" in consent_source or "risk" in consent_source.lower()
if has_risk_levels:
    details.append("PASS: Risk level concept in consent UI")
else:
    passed = False
    details.append("FAIL: No risk level descriptions for approvals")

# Check 3: Token revocation exists (mistakes are reversible)
revoke_path = os.path.join(OAUTH3_DIR, "revocation.py")
has_revocation = os.path.exists(revoke_path)
if has_revocation:
    rev_source = Path(revoke_path).read_text(encoding="utf-8")
    if "revoke" in rev_source:
        details.append("PASS: Token revocation exists (mistakes reversible)")
    else:
        passed = False
        details.append("FAIL: revocation.py exists but no revoke function")
else:
    passed = False
    details.append("FAIL: revocation.py missing entirely")

# Check 4: Step-up auth for high-risk actions (kids need extra protection)
step_up_path = os.path.join(OAUTH3_DIR, "step_up.py")
has_step_up = os.path.exists(step_up_path)
if has_step_up:
    step_source = Path(step_up_path).read_text(encoding="utf-8")
    if "step_up" in step_source.lower() or "nonce" in step_source:
        details.append("PASS: Step-up authentication exists for high-risk actions")
    else:
        passed = False
        details.append("FAIL: step_up.py exists but lacks step-up logic")
else:
    passed = False
    details.append("FAIL: step_up.py missing -- no elevated auth for dangerous actions")

# Check 5: Scope descriptions are human-readable (not just code identifiers)
scopes_path = os.path.join(OAUTH3_DIR, "scopes.py")
if os.path.exists(scopes_path):
    scopes_source = Path(scopes_path).read_text(encoding="utf-8")
    if "description" in scopes_source.lower():
        details.append("PASS: Scope descriptions exist (human-readable approvals)")
    else:
        passed = False
        details.append("FAIL: Scopes lack human-readable descriptions")
else:
    passed = False
    details.append("FAIL: scopes.py missing")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 9: Recipe Library Teen Platform Coverage (Floor 37 Kenji, Floor 40 Yara, Floor 34 Theo) ===
# Q: Does the recipe library serve professional workflows but miss teen platforms entirely?
# Q: Are there Discord, TikTok, Instagram, Spotify, Twitch, Steam recipe templates?
# Q: Does the product know that teens create on TikTok, not LinkedIn?
# Check: scan data/default/recipes/ for teen vs professional platform coverage

probe_name = "recipe_teen_platforms"
passed = True
details = []

teen_platforms = ["discord", "tiktok", "instagram", "spotify", "twitch", "steam", "youtube"]
pro_platforms = ["gmail", "linkedin", "github", "slack", "notion", "substack"]

teen_found = []
pro_found = []

# Walk recipes directory
for root, dirs, files in os.walk(RECIPES_DIR):
    for f in files:
        if f.endswith(".json"):
            fname_lower = f.lower()
            path_lower = os.path.join(root, f).lower()
            for platform in teen_platforms:
                if platform in fname_lower or platform in path_lower:
                    if platform not in teen_found:
                        teen_found.append(platform)
            for platform in pro_platforms:
                if platform in fname_lower or platform in path_lower:
                    if platform not in pro_found:
                        pro_found.append(platform)

details.append(f"Professional platforms with recipes: {pro_found} ({len(pro_found)}/{len(pro_platforms)})")
details.append(f"Teen platforms with recipes: {teen_found} ({len(teen_found)}/{len(teen_platforms)})")

# Count total recipes
total_recipes = sum(1 for _, _, files in os.walk(RECIPES_DIR) for f in files if f.endswith(".json") and "primewiki" not in _)

if len(teen_found) == 0:
    passed = False
    details.append("FAIL: ZERO teen platform recipes -- library is adult-professional only")
elif len(teen_found) < 3:
    passed = False
    details.append(f"FAIL: Only {len(teen_found)} teen platform(s) covered -- insufficient")
else:
    details.append(f"PASS: {len(teen_found)} teen platforms have recipe coverage")

missing = [p for p in teen_platforms if p not in teen_found]
if missing:
    details.append(f"MISSING teen platforms: {missing}")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 10: Gamification Language (Floor 2 Lucas, Floor 1 Zara) ===
# Q: Is there gamification language (achievements, levels, progress) that speaks to gamers?
# Q: Does the YinYang approval dialog read as a game mechanic?
# Check: web pages and JS for gamification vocabulary

probe_name = "gamification_language"
passed = True
details = []

gamification_terms = [
    "achievement", "level", "badge", "progress", "score",
    "streak", "unlock", "complete", "reward", "quest",
]

term_hits = Counter()

for html_file in HTML_FILES:
    html_path = os.path.join(WEB_DIR, html_file)
    if os.path.exists(html_path):
        content = Path(html_path).read_text(encoding="utf-8", errors="ignore").lower()
        for term in gamification_terms:
            if term in content:
                term_hits[term] += 1

# Also check JS files
js_dir = os.path.join(WEB_DIR, "js")
if os.path.isdir(js_dir):
    for js_file in os.listdir(js_dir):
        if js_file.endswith(".js"):
            content = Path(os.path.join(js_dir, js_file)).read_text(encoding="utf-8", errors="ignore").lower()
            for term in gamification_terms:
                if term in content:
                    term_hits[term] += 1

found_terms = list(term_hits.keys())
details.append(f"Gamification terms found: {found_terms} ({len(found_terms)}/{len(gamification_terms)})")

if len(found_terms) < 3:
    passed = False
    details.append(f"FAIL: Only {len(found_terms)} gamification terms -- not engaging for gamer teens")
else:
    details.append(f"PASS: {len(found_terms)} gamification terms present in UI")

# Check: YinYang tutorial for game-like onboarding
tutorial_path = os.path.join(WEB_DIR, "js", "yinyang-tutorial-v2.js")
if os.path.exists(tutorial_path):
    tut_source = Path(tutorial_path).read_text(encoding="utf-8", errors="ignore").lower()
    game_terms_in_tutorial = [t for t in gamification_terms if t in tut_source]
    if game_terms_in_tutorial:
        details.append(f"PASS: Tutorial uses gamification terms: {game_terms_in_tutorial}")
    else:
        details.append("INFO: Tutorial exists but lacks gamification vocabulary")
else:
    details.append("INFO: No v2 tutorial JS found")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 11: Local-First + Offline Capability (Floor 15 Lily, Floor 25 Fatima, Floor 32 Caleb) ===
# Q: Does the local-first architecture work on a 4G-only rural connection?
# Q: Can the product be installed without reliable broadband?
# Q: Does local-first fail rural youth at installation?
# Check: server runs locally on port 9222, settings are local, no cloud-required startup

probe_name = "local_first_offline"
passed = True
details = []

server_path = os.path.join(PROJECT_ROOT, "solace_browser_server.py")
assert os.path.exists(server_path), "solace_browser_server.py not found"

server_source = Path(server_path).read_text(encoding="utf-8")

# Check 1: Server binds to local port (not cloud-only)
has_local_port = "9222" in server_source or "port" in server_source.lower()
if has_local_port:
    details.append("PASS: Local port binding found (9222)")
else:
    passed = False
    details.append("FAIL: No local port binding -- cloud-only?")

# Check 2: Settings stored locally (not cloud-required)
settings_path = os.path.join(SRC_DIR, "settings_manager.py")
settings_source = Path(settings_path).read_text(encoding="utf-8")
if ".solace" in settings_source or "settings.json" in settings_source:
    details.append("PASS: Settings stored locally (~/.solace/)")
else:
    passed = False
    details.append("FAIL: Settings not stored locally")

# Check 3: No mandatory cloud call in startup path
# Parse server AST for cloud URL requirements in __init__ or startup
tree = ast.parse(server_source)
cloud_required_in_init = False
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef) and node.name in ("__init__", "start", "main"):
        func_source = ast.get_source_segment(server_source, node) or ""
        if re.search(r"requests\.get|requests\.post|aiohttp\.ClientSession", func_source):
            cloud_required_in_init = True

if cloud_required_in_init:
    details.append("INFO: Cloud call found in startup -- may block offline users")
else:
    details.append("PASS: No mandatory cloud call in server startup")

# Check 4: Evidence chain works locally
evidence_init = os.path.join(SRC_DIR, "evidence", "__init__.py")
if os.path.exists(evidence_init):
    details.append("PASS: Evidence module exists for local evidence storage")
else:
    passed = False
    details.append("FAIL: Evidence module missing")

# Check 5: Recipes are bundled locally
local_recipes = sum(1 for _, _, files in os.walk(RECIPES_DIR) for f in files if f.endswith(".json"))
if local_recipes > 10:
    details.append(f"PASS: {local_recipes} recipes bundled locally (no download needed)")
else:
    passed = False
    details.append(f"FAIL: Only {local_recipes} local recipes -- insufficient offline value")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 12: Shared Device + Profile Isolation (Floor 28 Leo, Floor 14 Sana) ===
# Q: Does the product support multi-user profile switching for shared devices?
# Q: Can Leo's family members see what he is doing (privacy issue on shared device)?
# Q: Does the product support family-shared accounts or multi-user profiles?
# Check: profiles/manager.py for multi-profile support + session isolation

probe_name = "profile_isolation"
passed = True
details = []

profile_source = Path(os.path.join(SRC_DIR, "profiles", "manager.py")).read_text(encoding="utf-8")

# Check 1: Multiple profiles supported
has_multi_profile = "MAX_PROFILES" in profile_source or "profiles" in profile_source.lower()
if has_multi_profile:
    details.append("PASS: Multi-profile concept exists")
else:
    passed = False
    details.append("FAIL: No multi-profile support")

# Check 2: Profile isolation error class exists
has_isolation = "ProfileIsolationError" in profile_source
if has_isolation:
    details.append("PASS: ProfileIsolationError defined -- cross-profile access blocked")
else:
    passed = False
    details.append("FAIL: No profile isolation enforcement")

# Check 3: Session manager has per-session data dirs
session_mgr_path = os.path.join(SRC_DIR, "session_manager.py")
session_source = Path(session_mgr_path).read_text(encoding="utf-8")

has_session_dirs = "session_id" in session_source and ("sessions" in session_source or "data_dir" in session_source.lower())
if has_session_dirs:
    details.append("PASS: Session-specific data directories exist")
else:
    passed = False
    details.append("FAIL: No per-session data directory isolation")

# Check 4: Incognito/guest mode
has_incognito = "incognito" in session_source.lower()
if has_incognito:
    details.append("PASS: Incognito/guest profile exists for shared device privacy")
else:
    passed = False
    details.append("FAIL: No incognito/guest mode for shared devices")

# Check 5: OAuth3 token is per-profile (not global)
has_token_per_profile = "oauth3_token_id" in profile_source or "token_id" in profile_source
if has_token_per_profile:
    details.append("PASS: OAuth3 token bound per profile")
else:
    passed = False
    details.append("FAIL: No per-profile token binding")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 13: Plugin Sandbox Safety (Floor 48 Dragon Rider, Floor 4 Jayden) ===
# Q: Is the product safe for a 15-year-old to use unsupervised?
# Q: Would a parent who discovers their 10-year-old using this product be alarmed?
# Check: plugin sandbox enforces scope limits, no eval/exec, resource limits

probe_name = "plugin_sandbox_safety"
passed = True
details = []

sandbox_path = os.path.join(SRC_DIR, "plugins", "sandbox.py")
if not os.path.exists(sandbox_path):
    passed = False
    details.append("FAIL: plugins/sandbox.py does not exist -- no plugin isolation")
else:
    sandbox_source = Path(sandbox_path).read_text(encoding="utf-8")

    # Check 1: No exec/eval allowed
    bans_exec = "exec" in sandbox_source and ("no exec" in sandbox_source.lower() or "forbidden" in sandbox_source.lower() or "eval" in sandbox_source)
    if bans_exec:
        details.append("PASS: exec/eval mentioned in sandbox constraints")
    else:
        details.append("INFO: exec/eval ban not explicitly documented in sandbox")

    # Check 2: Scope-limited execution
    has_scope_check = "scope" in sandbox_source.lower()
    if has_scope_check:
        details.append("PASS: Scope-limited execution in sandbox")
    else:
        passed = False
        details.append("FAIL: No scope enforcement in plugin sandbox")

    # Check 3: Resource limits (memory, CPU, output)
    has_resource_limits = "max_memory" in sandbox_source or "resource" in sandbox_source.lower()
    if has_resource_limits:
        details.append("PASS: Resource limits in sandbox (memory/CPU/output)")
    else:
        passed = False
        details.append("FAIL: No resource limits in plugin sandbox")

    # Check 4: Kill switch exists
    has_kill = "terminate" in sandbox_source or "kill" in sandbox_source
    if has_kill:
        details.append("PASS: Kill switch (terminate) exists")
    else:
        passed = False
        details.append("FAIL: No kill switch in sandbox")

    # Check 5: SandboxViolationError for unauthorized access
    has_violation_error = "SandboxViolationError" in sandbox_source
    if has_violation_error:
        details.append("PASS: SandboxViolationError defined for access violations")
    else:
        passed = False
        details.append("FAIL: No violation error class")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 14: Evidence Chain as Learning Tool (Floor 3 Sofia, Floor 42 Noor, Floor 43 Arjun) ===
# Q: Do evidence chains map to source tracking that a teen journalist would use?
# Q: Is the evidence chain a proof log that maps to mathematical reasoning?
# Q: Does the evidence chain serve as a research log for privacy-aware teens?
# Check: evidence module uses SHA-256 hash chains, readable summaries

probe_name = "evidence_chain_teens"
passed = True
details = []

evidence_init = os.path.join(SRC_DIR, "evidence", "__init__.py")
assert os.path.exists(evidence_init), "evidence/__init__.py missing"

evidence_source = Path(evidence_init).read_text(encoding="utf-8")

# Check 1: SHA-256 hash chain
has_sha256 = "sha256" in evidence_source.lower() or "hashlib" in evidence_source
if has_sha256:
    details.append("PASS: SHA-256 hash chain in evidence module")
else:
    passed = False
    details.append("FAIL: No SHA-256 hash chain -- evidence not tamper-evident")

# Check 2: Summary formatter exists (human-readable evidence)
summary_path = os.path.join(SRC_DIR, "evidence", "summary_formatter.py")
if os.path.exists(summary_path):
    summary_source = Path(summary_path).read_text(encoding="utf-8")
    has_format = "format" in summary_source.lower() or "summary" in summary_source.lower()
    if has_format:
        details.append("PASS: Evidence summary formatter exists (human-readable for teens)")
    else:
        passed = False
        details.append("FAIL: Summary formatter exists but has no formatting logic")
else:
    passed = False
    details.append("FAIL: No evidence summary formatter -- raw hashes only")

# Check 3: Evidence is append-only (tamper-evident)
has_append_only = "append" in evidence_source.lower() or "chain" in evidence_source.lower()
if has_append_only:
    details.append("PASS: Evidence chain is append-based (tamper-evident)")
else:
    details.append("INFO: Append-only nature not explicit in evidence __init__")

# Check 4: Audit chain module exists (deeper evidence)
audit_chain_path = os.path.join(SRC_DIR, "audit", "chain.py")
if os.path.exists(audit_chain_path):
    audit_source = Path(audit_chain_path).read_text(encoding="utf-8")
    has_chain = "chain" in audit_source.lower() and "hash" in audit_source.lower()
    if has_chain:
        details.append("PASS: Audit chain module with hash chaining exists")
    else:
        details.append("INFO: Audit chain.py exists but hash chaining not confirmed")
else:
    details.append("INFO: No audit/chain.py module")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 15: Privacy + Data Sovereignty (Floor 19 Maya, Floor 47 Pax, Floor 42 Noor) ===
# Q: Can Maya know exactly where her data goes, who owns it, and if it is used for training?
# Q: Is there an explicit privacy-by-design statement?
# Q: Does the product have a clear position on digital rights?
# Check: privacy settings page, local-first messaging, no-tracking cookies

probe_name = "privacy_data_sovereignty"
passed = True
details = []

# Check 1: Settings page has privacy section
settings_html = os.path.join(WEB_DIR, "settings.html")
if os.path.exists(settings_html):
    settings_content = Path(settings_html).read_text(encoding="utf-8")
    has_privacy_section = 'id="privacy"' in settings_content or "Privacy" in settings_content
    if has_privacy_section:
        details.append("PASS: Settings page has privacy section")
    else:
        passed = False
        details.append("FAIL: No privacy section in settings")

    # Check local-first messaging
    has_local_first_msg = "local-first" in settings_content.lower() or "data stays on your machine" in settings_content.lower()
    if has_local_first_msg:
        details.append("PASS: Local-first data sovereignty message present")
    else:
        passed = False
        details.append("FAIL: No local-first data messaging")
else:
    passed = False
    details.append("FAIL: settings.html missing")

# Check 2: Cookie consent banner (no tracking)
footer_path = os.path.join(WEB_DIR, "partials-footer.html")
if os.path.exists(footer_path):
    footer_content = Path(footer_path).read_text(encoding="utf-8")
    has_cookie_banner = "cookie" in footer_content.lower()
    no_tracking = "do not use tracking" in footer_content.lower() or "no tracking" in footer_content.lower()
    if has_cookie_banner:
        details.append("PASS: Cookie consent banner present")
    else:
        details.append("INFO: No cookie banner in footer")
    if no_tracking:
        details.append("PASS: Explicit no-tracking cookies declaration")
    else:
        details.append("INFO: No explicit no-tracking declaration")

# Check 3: Privacy policy link exists
privacy_link_found = False
for html_file in ["partials-footer.html", "start.html", "settings.html"]:
    html_path = os.path.join(WEB_DIR, html_file)
    if os.path.exists(html_path):
        content = Path(html_path).read_text(encoding="utf-8")
        if "privacy" in content.lower() and ("href" in content or "link" in content.lower()):
            privacy_link_found = True
            break

if privacy_link_found:
    details.append("PASS: Privacy policy link found in web pages")
else:
    passed = False
    details.append("FAIL: No privacy policy link in any page")

# Check 4: No GA/analytics tracking scripts
tracking_found = False
for html_file in HTML_FILES:
    html_path = os.path.join(WEB_DIR, html_file)
    if os.path.exists(html_path):
        content = Path(html_path).read_text(encoding="utf-8")
        if re.search(r"google-analytics|gtag|GA4|facebook.*pixel|hotjar", content, re.I):
            tracking_found = True
            details.append(f"FAIL: Tracking script found in {html_file}")
            break

if not tracking_found:
    details.append("PASS: No analytics/tracking scripts in any HTML page")
else:
    passed = False

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === PROBE 16: Onboarding + Discoverability (Floor 1 Zara, Floor 7 Ryan, Floor 49 Tyson) ===
# Q: Is the onboarding path discoverable for a teen who already lives in terminals?
# Q: Do video tutorials exist showing what automation looks like in practice?
# Q: Can you explain what this does without using 'AI,' 'automation,' or 'recipe'?
# Check: onboarding JS exists, quick-start docs, demo page, tutorial system

probe_name = "onboarding_discoverability"
passed = True
details = []

# Check 1: Onboarding JS exists
onboarding_path = os.path.join(WEB_DIR, "js", "onboarding.js")
if os.path.exists(onboarding_path):
    onb_source = Path(onboarding_path).read_text(encoding="utf-8")
    has_steps = "step" in onb_source.lower() or "wizard" in onb_source.lower() or "tour" in onb_source.lower()
    if has_steps:
        details.append("PASS: Onboarding JS with step/wizard/tour flow")
    else:
        details.append("INFO: onboarding.js exists but no step flow detected")
else:
    passed = False
    details.append("FAIL: No onboarding.js -- no guided first-run experience")

# Check 2: Quick-start documentation
quick_start = os.path.join(WEB_DIR, "docs", "quick-start.html")
if os.path.exists(quick_start):
    qs_content = Path(quick_start).read_text(encoding="utf-8")
    word_count = len(qs_content.split())
    details.append(f"PASS: Quick-start guide exists ({word_count} words)")
else:
    passed = False
    details.append("FAIL: No quick-start guide")

# Check 3: Demo page exists
demo_path = os.path.join(WEB_DIR, "demo.html")
if os.path.exists(demo_path):
    demo_content = Path(demo_path).read_text(encoding="utf-8")
    has_interactive = "interactive" in demo_content.lower() or "preview" in demo_content.lower() or "try" in demo_content.lower()
    if has_interactive:
        details.append("PASS: Demo page has interactive/preview elements")
    else:
        details.append("INFO: Demo page exists but may not be interactive")
else:
    passed = False
    details.append("FAIL: No demo page")

# Check 4: Tutorial system exists (for Ryan who prefers videos/visual)
tutorial_exists = False
for js_file in ["yinyang-tutorial.js", "yinyang-tutorial-v2.js"]:
    tut_path = os.path.join(WEB_DIR, "js", js_file)
    if os.path.exists(tut_path):
        tutorial_exists = True
        details.append(f"PASS: Tutorial JS found ({js_file})")

if not tutorial_exists:
    passed = False
    details.append("FAIL: No tutorial system for visual learners")

# Check 5: Start page for new users
start_path = os.path.join(WEB_DIR, "start.html")
if os.path.exists(start_path):
    details.append("PASS: start.html exists for new user onboarding")
else:
    passed = False
    details.append("FAIL: No start page for new users")

# Check 6: Error messages are understandable (scan for jargon-heavy errors)
jargon_errors = 0
for root, dirs, files in os.walk(SRC_DIR):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    for f in files:
        if f.endswith(".py"):
            content = Path(os.path.join(root, f)).read_text(encoding="utf-8", errors="ignore")
            # Look for raw exception messages with hex/hash jargon
            hex_errors = re.findall(r'raise\s+\w+Error\(["\'].*[0-9a-f]{32}', content)
            jargon_errors += len(hex_errors)

if jargon_errors > 5:
    details.append(f"INFO: {jargon_errors} error messages contain hex/hash jargon -- may confuse teens")
else:
    details.append(f"PASS: Few jargon-heavy error messages ({jargon_errors})")

results[probe_name] = {"pass": passed, "detail": "; ".join(details)}
status = "PASS" if passed else "FAIL"
print(f"{status}: {probe_name}")
for d in details:
    print(f"  {d}")

In [ ]:
# === EVIDENCE SUMMARY ===
# Aggregate all probe results into a final score

total = len(results)
passed_count = sum(1 for r in results.values() if r["pass"])
failed_count = total - passed_count
score = round((passed_count / total) * 100, 1) if total > 0 else 0
verdict = "PASS" if score >= MIN_SCORE else "FAIL"

print("=" * 70)
print(f"TOWER {TOWER}: {TOWER_NAME} -- QA SUMMARY")
print(f"DNA: nextgen(product) = intuition(no_manual) x safety(absolute) x fun(genuine) x growth(learning)")
print("=" * 70)
print(f"Total probes: {total}")
print(f"Passed: {passed_count}")
print(f"Failed: {failed_count}")
print(f"Score: {score}%")
print(f"Minimum: {MIN_SCORE}%")
print(f"Verdict: {verdict}")
print("=" * 70)

print("\nDETAILED RESULTS:")
for probe, result in results.items():
    status = "PASS" if result["pass"] else "FAIL"
    print(f"  [{status}] {probe}")

print("\nFAILED PROBES:")
for probe, result in results.items():
    if not result["pass"]:
        print(f"  [{probe}] {result['detail']}")

# Evidence JSON
evidence = {
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "northstar": NORTHSTAR,
    "total_probes": total,
    "passed": passed_count,
    "failed": failed_count,
    "score_pct": score,
    "min_score_pct": MIN_SCORE,
    "verdict": verdict,
    "probes": {
        name: {"pass": r["pass"], "detail": r["detail"]}
        for name, r in results.items()
    },
}

print("\nEVIDENCE JSON:")
print(json.dumps(evidence, indent=2))